# 06 — Bayesian Network / Causal Discovery

Learns a probabilistic directed graph over climate and health variables. Unlike ARM, BN captures **conditional independence** (variable A predicts B even after controlling for C) and suggests **causal direction** via temporal constraints.

**Workflow**:
1. Select 8–12 variables most predictive from notebooks 04 + 05
2. Learn structure with PC algorithm (constraint-based) + BIC scoring
3. Enforce directionality: prior-day climate → same-day health (tier constraint)
4. Bootstrap structure for stability (keep edges present in ≥70% of resamples)
5. Estimate CPDs (conditional probability distributions)
6. Query: P(breathing_elevated | extreme_heat, o3_high)
7. Compare with ARM lift values as cross-validation

**Input**: `data/processed/daily_merged.csv`  
**Output**: `data/processed/bn_stable_edges.csv`, network plots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    import bnlearn as bn
    BNLEARN_OK = True
except ImportError:
    BNLEARN_OK = False
    print('bnlearn not installed — run: conda run -n test pip install bnlearn')

PROC_DIR = Path('../data/processed')
print(f'bnlearn available: {BNLEARN_OK}')

In [ ]:
merged_path = PROC_DIR / 'daily_merged.csv'
if not merged_path.exists():
    raise FileNotFoundError('Run notebook 03 first to produce daily_merged.csv')

df = pd.read_csv(merged_path, parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)
print(f'Loaded: {df.shape}  |  {df.date.min().date()} → {df.date.max().date()}')

## 1. Variable Selection

Bayesian Network structure learning is sensitive to dimensionality. With 3,050 observations, limit to ≤12 variables total. Select based on:
- Top associations from notebook 04 (NegBin coefficients)
- Top antecedents from notebook 05 (ARM lift)
- Keep at least one negative control for validation

In [ ]:
# Core climate variables (select those available in the merged dataset)
CLIMATE_VARS = [v for v in ['temp_max', 'relative_humidity', 'precip_mm',
                              'wind_speed', 'pm25', 'o3']
                if v in df.columns]

# Health outcomes — prioritise high-volume, high climate-sensitivity
HEALTH_VARS = [v for v in [
    'code_06_breathing',
    'code_31_loss_of_consciousness',
    'code_09_cardiac_arrest',
    'code_28_stroke_tia',
    'code_17_falls',
    'mission_count_ems_critical',
] if v in df.columns][:4]  # cap at 4 to keep total ≤10

# One negative control
CONTROL_VARS = [v for v in ['ctrl_27_stabbing_gunshot'] if v in df.columns]

ALL_VARS = CLIMATE_VARS + HEALTH_VARS + CONTROL_VARS
print(f'Variables selected: {len(ALL_VARS)}')
print(f'  Climate : {CLIMATE_VARS}')
print(f'  Health  : {HEALTH_VARS}')
print(f'  Control : {CONTROL_VARS}')

## 2. Discretisation

Bayesian Networks with discrete nodes (discrete BN) are more interpretable and robust than continuous Gaussian BNs at this sample size.

In [ ]:
CLIMATE_BINS = {
    'temp_max':           ([-np.inf, 10, 20, 28, np.inf],    ['cold','cool','warm','hot']),
    'relative_humidity':  ([-np.inf, 50, 70, np.inf],         ['dry','moderate','humid']),
    'precip_mm':          ([-np.inf, 0.5, 10, np.inf],        ['dry','light','heavy']),
    'wind_speed':         ([-np.inf, 4, 10, np.inf],           ['calm','moderate','strong']),
    'pm25':               ([-np.inf, 10, 25, np.inf],          ['good','moderate','poor']),
    'o3':                 ([-np.inf, 60, 120, np.inf],         ['low','moderate','high']),
}

data_bn = df[ALL_VARS].copy()

# Discretise climate vars
for col, (bins, labels) in CLIMATE_BINS.items():
    if col in data_bn.columns:
        data_bn[col] = pd.cut(data_bn[col], bins=bins, labels=labels).astype(str)

# Discretise health vars into tertiles (low / normal / high)
for col in HEALTH_VARS + CONTROL_VARS:
    if col not in data_bn.columns:
        continue
    q33, q67 = data_bn[col].quantile([0.33, 0.67])
    short = col.replace('code_','').replace('ctrl_','ctrl_').replace('mission_count_','')
    data_bn[col] = pd.cut(
        data_bn[col],
        bins=[-np.inf, q33, q67, np.inf],
        labels=[f'{short}_low', f'{short}_mid', f'{short}_high']
    ).astype(str)

data_bn = data_bn.replace('nan', np.nan).dropna()
print(f'BN dataset: {data_bn.shape}')
print('Unique values per variable:')
for c in ALL_VARS:
    if c in data_bn.columns:
        print(f'  {c}: {sorted(data_bn[c].unique())}')

## 3. Temporal Lag — Prior-Day Climate → Same-Day Health

We use prior-day climate values (shifted by 1 day) to enforce temporal causality: past climate predicts future health, not the reverse.

In [ ]:
# Build lagged dataset: climate_t-1 → health_t
lag_df = pd.DataFrame(index=df.index)

# Lagged climate (day t-1)
for col, (bins, labels) in CLIMATE_BINS.items():
    if col not in df.columns:
        continue
    discretised = pd.cut(df[col], bins=bins, labels=labels).astype(str)
    lag_df[f'{col}_lag1'] = discretised.shift(1)

# Same-day health
for col in HEALTH_VARS + CONTROL_VARS:
    if col in data_bn.columns:
        lag_df[col] = data_bn[col].values

lag_df = lag_df.replace('nan', np.nan).dropna()

LAG_CLIMATE_VARS = [f'{v}_lag1' for v in CLIMATE_VARS if f'{v}_lag1' in lag_df.columns]
ALL_VARS_LAGGED  = LAG_CLIMATE_VARS + HEALTH_VARS + CONTROL_VARS
lag_df = lag_df[ALL_VARS_LAGGED]

print(f'Lagged BN dataset: {lag_df.shape}')
print(f'Climate (lag-1): {LAG_CLIMATE_VARS}')
print(f'Health (same day): {HEALTH_VARS}')

## 4. Structure Learning — PC Algorithm

PC (Peter-Clark) algorithm:
1. Start with a fully connected undirected graph
2. Remove edges where variables are conditionally independent (chi-square test)
3. Orient edges based on v-structures and acyclicity constraints
4. Apply tier constraint: climate_lag1 nodes cannot receive edges from health nodes

In [ ]:
if not BNLEARN_OK:
    print('Install bnlearn first: conda run -n test pip install bnlearn')
else:
    # White-list: climate_lag → health edges are allowed
    # Black-list: health → climate_lag edges are forbidden
    blacklist = []
    for health_var in HEALTH_VARS + CONTROL_VARS:
        for clim_var in LAG_CLIMATE_VARS:
            blacklist.append((health_var, clim_var))  # health cannot cause prior-day climate

    print(f'Blacklisted edges (health → climate): {len(blacklist)}')
    print('Learning structure with PC algorithm...')

    model = bn.structure_learning.fit(
        lag_df,
        methodtype='pc',
        scoretype='bic',
        black_list=blacklist if blacklist else None,
        verbose=0
    )

    edges = model['model'].edges()
    print(f'\nLearned edges ({len(list(edges))}):')
    for e in sorted(model['model'].edges()):
        print(f'  {e[0]} → {e[1]}')

## 5. Bootstrap Stability Analysis

Resample the data 100 times and relearn structure each time. Keep only edges present in ≥70% of bootstraps — these are structurally stable.

In [ ]:
N_BOOTSTRAP = 100
STABILITY_THRESHOLD = 0.70

if BNLEARN_OK:
    from collections import defaultdict
    edge_counts = defaultdict(int)

    np.random.seed(42)
    for i in range(N_BOOTSTRAP):
        boot = lag_df.sample(n=len(lag_df), replace=True)
        try:
            m = bn.structure_learning.fit(
                boot, methodtype='pc', scoretype='bic',
                black_list=blacklist if blacklist else None,
                verbose=0
            )
            for e in m['model'].edges():
                edge_counts[e] += 1
        except Exception:
            pass
        if (i + 1) % 20 == 0:
            print(f'  Bootstrap {i+1}/{N_BOOTSTRAP}', flush=True)

    # Filter stable edges
    stable_edges = [
        (src, dst, cnt / N_BOOTSTRAP)
        for (src, dst), cnt in edge_counts.items()
        if cnt / N_BOOTSTRAP >= STABILITY_THRESHOLD
    ]
    stable_edges.sort(key=lambda x: -x[2])

    print(f'\nStable edges (≥{STABILITY_THRESHOLD*100:.0f}% of bootstraps): {len(stable_edges)}')
    for src, dst, prob in stable_edges:
        print(f'  {src} → {dst}  (stability={prob:.2f})')

    stable_df = pd.DataFrame(stable_edges, columns=['source', 'target', 'stability'])
    stable_df.to_csv(PROC_DIR / 'bn_stable_edges.csv', index=False)
    print(f'\nSaved: bn_stable_edges.csv')

## 6. Build Final Model on Stable Edges & Estimate CPDs

In [ ]:
if BNLEARN_OK and 'stable_edges' in dir() and len(stable_edges) > 0:
    # Build DAG from stable edges only
    edges_list = [(src, dst) for src, dst, _ in stable_edges]
    final_model = bn.make_DAG(edges_list)

    # Fit CPDs using full dataset
    final_model = bn.parameter_learning.fit(final_model, lag_df, methodtype='bayes')
    print('CPDs estimated on stable graph')
    print(f'Final model edges: {len(final_model["model"].edges())}')

## 7. Inference — Conditional Probability Queries

Query the learned BN to answer specific climate-health questions.

In [ ]:
if BNLEARN_OK and 'final_model' in dir():
    # Helper: query P(health_outcome | climate_conditions)
    def query_bn(model, target_var, evidence_dict):
        try:
            q = bn.inference.fit(
                model,
                variables=[target_var],
                evidence=evidence_dict,
                verbose=0
            )
            return q['p'] if hasattr(q, 'p') else q
        except Exception as e:
            return f'Query failed: {e}'

    queries = []

    # Q1: P(breathing | temp_hot, o3_high)
    if 'code_06_breathing' in HEALTH_VARS and 'temp_max_lag1' in LAG_CLIMATE_VARS:
        target = 'code_06_breathing'
        short  = target.replace('code_','')
        print(f'Q1: P({target} | temp_max=hot, o3=high)')
        result = query_bn(final_model, target,
                          {'temp_max_lag1': 'hot', 'o3_lag1': 'high'})
        print(result)
        queries.append({'query': 'breathing | hot + o3_high', 'result': str(result)})

    # Q2: P(falls | temp_cold, precip_heavy)
    if 'code_17_falls' in HEALTH_VARS and 'temp_max_lag1' in LAG_CLIMATE_VARS:
        target = 'code_17_falls'
        print(f'\nQ2: P({target} | temp_max=cold, precip=heavy)')
        result = query_bn(final_model, target,
                          {'temp_max_lag1': 'cold', 'precip_mm_lag1': 'heavy'})
        print(result)
        queries.append({'query': 'falls | cold + heavy_precip', 'result': str(result)})

    # Q3: P(cardiac_arrest | temp_hot)
    if 'code_09_cardiac_arrest' in HEALTH_VARS and 'temp_max_lag1' in LAG_CLIMATE_VARS:
        target = 'code_09_cardiac_arrest'
        print(f'\nQ3: P({target} | temp_max=hot) vs P({target} | temp_max=cool)')
        r_hot  = query_bn(final_model, target, {'temp_max_lag1': 'hot'})
        r_cool = query_bn(final_model, target, {'temp_max_lag1': 'cool'})
        print('Hot:', r_hot)
        print('Cool:', r_cool)
        queries.append({'query': 'cardiac | hot vs cool', 'result': f'hot={r_hot} cool={r_cool}'})

    # Q4: Negative control — P(stabbing | temp_hot) — should be ≈ marginal
    if 'ctrl_27_stabbing_gunshot' in CONTROL_VARS and 'temp_max_lag1' in LAG_CLIMATE_VARS:
        target = 'ctrl_27_stabbing_gunshot'
        print(f'\nQ4 (negative control): P({target} | temp_max=hot)')
        result = query_bn(final_model, target, {'temp_max_lag1': 'hot'})
        print(result)
        queries.append({'query': 'stabbing | hot (CONTROL)', 'result': str(result)})

    pd.DataFrame(queries).to_csv(PROC_DIR / 'bn_queries.csv', index=False)
    print('\nQueries saved to bn_queries.csv')

## 8. Cross-Validation Against ARM Rules

For each stable BN edge (climate → health), look up the corresponding ARM lift. High lift + stable BN edge = strong, consistent non-linear association.

In [ ]:
arm_path = PROC_DIR / 'arm_rules_stable.csv'
if arm_path.exists() and 'stable_df' in dir():
    arm_rules = pd.read_csv(arm_path)

    # Match BN edges to ARM rules by outcome variable name
    print('BN edge vs ARM lift cross-reference:')
    print(f'{"BN Edge":<55} {"Best ARM Lift":>12}')
    print('-' * 70)
    for _, row in stable_df.iterrows():
        src = row['source'].replace('_lag1', '')
        dst = row['target']
        # Search ARM rules where antecedent contains source variable name
        # and consequent relates to destination variable
        dst_short = dst.replace('code_','').replace('mission_count_','')
        matches = arm_rules[
            arm_rules['antecedent_str'].str.contains(src, na=False) &
            arm_rules['consequent_str'].str.contains(dst_short[:10], na=False)
        ]
        best_lift = matches['lift'].max() if len(matches) > 0 else np.nan
        print(f'{row["source"]:30s} → {row["target"]:20s}  lift={best_lift:.2f}' if not np.isnan(best_lift)
              else f'{row["source"]:30s} → {row["target"]:20s}  lift=no match')
else:
    print('Run notebooks 05 and bootstrap step first.')

## 9. Network Visualisation

In [ ]:
if BNLEARN_OK and 'stable_df' in dir() and len(stable_df) > 0:
    try:
        import networkx as nx

        G = nx.DiGraph()
        for _, row in stable_df.iterrows():
            G.add_edge(row['source'], row['target'], weight=row['stability'])

        # Node colours: climate=steelblue, health=tomato, control=gray
        node_colors = []
        for node in G.nodes():
            if any(node.startswith(v) for v in ['temp','precip','wind','pm','o3','no2','relative']):
                node_colors.append('steelblue')
            elif node.startswith('ctrl_'):
                node_colors.append('lightgray')
            else:
                node_colors.append('tomato')

        # Edge widths proportional to stability
        edge_weights = [G[u][v]['weight'] * 4 for u, v in G.edges()]

        pos = nx.spring_layout(G, seed=42, k=2.5)
        short_labels = {n: n.replace('_lag1','\n(lag1)').replace('code_','').replace('ctrl_','ctrl:') for n in G.nodes()}

        fig, ax = plt.subplots(figsize=(14, 8))
        nx.draw_networkx(
            G, pos=pos, labels=short_labels,
            node_color=node_colors, node_size=1800,
            edge_color='dimgray', width=edge_weights,
            arrows=True, arrowsize=20,
            font_size=7, font_color='white',
            ax=ax
        )
        legend_items = [
            mpatches.Patch(color='steelblue', label='Climate (lag-1)'),
            mpatches.Patch(color='tomato',    label='Health outcome'),
            mpatches.Patch(color='lightgray', label='Negative control'),
        ]
        ax.legend(handles=legend_items, loc='upper left', fontsize=9)
        ax.set_title(f'Stable Bayesian Network (bootstrap ≥{STABILITY_THRESHOLD*100:.0f}%)\n'
                     f'Edge width = stability  |  {len(stable_df)} edges', fontsize=11)
        ax.axis('off')
        fig.tight_layout()
        plt.savefig(PROC_DIR / 'bayesian_network.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved: bayesian_network.png')
    except ImportError:
        print('networkx not available for plotting — pip install networkx')
        bn.plot(final_model)
else:
    print('No stable edges to plot.')